In [1]:
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, VotingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 1. Load the Iris dataset
iris = load_iris()
X = iris.data
y = iris.target

# 2. Split into training and testing data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Create the three models
ada = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),
    n_estimators=50,
    random_state=42
)

gbm = GradientBoostingClassifier(
    n_estimators=100, random_state=42
)

xgb = XGBClassifier(
    n_estimators=100,
    objective="multi:softprob",
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=2
)

models = {
    "AdaBoost": ada,
    "GBM": gbm,
    "XGBoost": xgb
}

# 4. Train and evaluate each model
results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    results.append([
        name,
        accuracy_score(y_test, pred),
        precision_score(y_test, pred, average="macro", zero_division=0),
        recall_score(y_test, pred, average="macro", zero_division=0),
        f1_score(y_test, pred, average="macro", zero_division=0)
    ])

# 5. Apply hard and soft voting
voting_models = {
    "Hard Voting": VotingClassifier(
        estimators=[("ada", ada), ("gbm", gbm), ("xgb", xgb)],
        voting="hard"
    ),
    "Soft Voting": VotingClassifier(
        estimators=[("ada", ada), ("gbm", gbm), ("xgb", xgb)],
        voting="soft"
    )
}

for name, model in voting_models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    results.append([
        name,
        accuracy_score(y_test, pred),
        precision_score(y_test, pred, average="macro", zero_division=0),
        recall_score(y_test, pred, average="macro", zero_division=0),
        f1_score(y_test, pred, average="macro", zero_division=0)
    ])

# 6. Display the comparison
result_df = pd.DataFrame(results, columns=[
    "Model", "Accuracy", "Precision", "Recall", "F1-score"
])

print(result_df.round(3))

         Model  Accuracy  Precision  Recall  F1-score
0     AdaBoost     0.933      0.933   0.933     0.933
1          GBM     0.967      0.970   0.967     0.967
2      XGBoost     0.933      0.933   0.933     0.933
3  Hard Voting     0.967      0.970   0.967     0.967
4  Soft Voting     0.967      0.970   0.967     0.967
